# Clase 028 — groupby (split-apply-combine)

**Parte 0** · VanderPlas cap. 3 § 3.9.

> 🎯 El patrón fundamental del análisis tabular. 4 métodos: agg, transform, filter, apply.

> ⏱️ ~90 min

## ⚙️ Setup

In [ ]:
import numpy as np
import pandas as pd
rng = np.random.default_rng(42)

# Mini-dataset penguin-like
df = pd.DataFrame({
    'species': ['Adelie']*5 + ['Chinstrap']*4 + ['Gentoo']*5,
    'sex'    : ['M','F','M','F','M', 'M','F','M','F',  'M','F','M','F','M'],
    'masa'   : [3750, 3800, 3650, 3900, 3700,  3500, 3400, 3600, 3550,  5050, 4800, 5200, 4900, 5100],
    'pico'   : [39.1, 39.5, 40.3, 38.8, 39.3,  46.5, 46.0, 46.8, 45.9,  48.6, 47.5, 49.0, 48.2, 48.8],
})
print(df.head())

## 1️⃣ Split-apply-combine

```
split:  divide el DataFrame por valores de una columna
apply:  aplica función a cada grupo
combine: junta los resultados
```

El objeto `GroupBy` no calcula nada hasta que llamas una operación (lazy):

In [ ]:
g = df.groupby('species')
print(f'tipo: {type(g).__name__}')
print(f'grupos: {list(g.groups.keys())}')
print(f'tamaño por grupo:')
print(g.size())

## 2️⃣ `agg` — reduce a una fila por grupo

In [ ]:
# Una sola función
print('media por species:')
print(g[['masa','pico']].mean().round(2))

# Dict de funciones distintas
print('\nagg con dict:')
print(g.agg({'masa': 'mean', 'pico': ['min','max']}).round(2))

# Funciones nombradas (named aggregation)
print('\nnamed aggregation:')
print(g.agg(
    masa_media=('masa', 'mean'),
    pico_max=('pico', 'max'),
    n=('masa', 'count'),
).round(2))

## 3️⃣ `transform` — misma shape, broadcast por grupo

Útil para crear features dentro de un grupo (z-score, ratio sobre el grupo, imputación).

In [ ]:
# z-score de masa POR ESPECIE
df['masa_z'] = g['masa'].transform(lambda s: (s - s.mean()) / s.std())
print(df.round(3))

# Verifica: cada grupo tiene media ≈ 0 y std ≈ 1
print('\nMedia z por species:')
print(df.groupby('species')['masa_z'].mean().round(3))
print('\nStd z por species:')
print(df.groupby('species')['masa_z'].std().round(3))

## 4️⃣ `filter` — conserva grupos completos

In [ ]:
# Solo species con más de 4 individuos
result = g.filter(lambda x: len(x) > 4)
print(result['species'].value_counts())

## 5️⃣ `apply` — flexible y lento

Úsalo cuando los 3 anteriores no alcanzan (típicamente cuando necesitas devolver un DataFrame por grupo).

In [ ]:
# El más pesado de cada species
top = g.apply(lambda x: x.nlargest(1, 'masa'), include_groups=False)
print(top)

## 6️⃣ Múltiples columnas de agrupación

In [ ]:
by_sex = df.groupby(['species', 'sex'])['masa'].mean().round(0)
print('media por species × sex (MultiIndex):')
print(by_sex)
print('\nunstack(sex) → wide:')
print(by_sex.unstack('sex'))

## 🧠 Cuándo cada método

| Método | Shape salida | Caso típico |
|---|---|---|
| `agg` | filas = #grupos | resumen estadístico |
| `transform` | filas = original | z-score, normalizar por grupo |
| `filter` | subset de original | excluir grupos pequeños/raros |
| `apply` | flexible | cuando los otros 3 no alcanzan |

## ✅ Checklist

- [ ] Entiendo split-apply-combine
- [ ] Uso named aggregation con `agg(...)`
- [ ] Sé cuándo `transform` (preserva shape) vs `agg` (reduce)
- [ ] Uso `filter` para excluir grupos enteros
- [ ] Reservo `apply` para casos que los 3 anteriores no resuelven

## 📝 Homework

Ver `README.md`. agg múltiple, transform z-score, filter por n, apply top-3.

## 🔗 Referencias

- VanderPlas cap. 3 § 3.9
- [pandas groupby](https://pandas.pydata.org/docs/user_guide/groupby.html)
- Wickham, *Split-apply-combine* (2011)

➡️ **Siguiente:** [029 — pivot tables y crosstab](../029-pandas-pivot-tables-y-crosstab/README.md)